In [2]:
import os
import sys
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import cmocean as cmo
import easygems.healpix as egh
import intake
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

warnings.filterwarnings(
    "ignore",
    message=".*The return type of `Dataset.dims` will be changed.*",
    category=FutureWarning,
)

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    sys.path.insert(1, "../onset_identification")
from utils import (
    domains,
    get_nn_lon_lat_index,
    haversine,
    hp_mods,
    hp_to_latlon,
    ifs_hp_mods,
    relon,
    sim_colors,
    sim_labels,
    sims,
)


def calculate_mse(
    temperature,
    geopotential_height,
    specific_humidity,
):
    import metpy.constants as mpconst

    g = mpconst.g.magnitude
    cp = mpconst.dry_air_spec_heat_press.magnitude
    Lv = mpconst.water_heat_vaporization.magnitude

    # compute h
    return cp * temperature + geopotential_height + Lv * specific_humidity


def mass_weighted_column_integral(da):
    import metpy.constants as mpconst

    g = mpconst.g.magnitude
    p = da["pressure"].values
    dp_da = xr.DataArray(
        np.abs(
            np.diff(
                np.concatenate(
                    (
                        [p[0] - (p[1] - p[0]) / 2],
                        0.5 * (p[1:] + p[:-1]),
                        [p[-1] + (p[-1] - p[-2]) / 2],
                    )
                )
            )
        ),
        coords={"pressure": p},
        dims=["pressure"],
    )

    int_da = (da * dp_da).sum("pressure") / g
    return int_da

In [3]:
url = "https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml"
cat = intake.open_catalog(url)["online"]
sim=sims[0]
zoom=3

sim_cat = cat[sim]

if "hk26" in sim:
    ds = sim_cat(zoom=zoom, time="PT3H").to_dask().pipe(hp_mods)
else:
    if "icon" in sim or "nicam" in sim:
        ds = sim_cat(zoom=zoom).to_dask().pipe(egh.attach_coords).pr
    elif "ifs" in sim:
        zoom = 7
        ds = sim_cat(zoom=zoom).to_dask().pipe(ifs_hp_mods)
    else:
        ds = sim_cat(zoom=zoom, time="PT3H").to_dask().pipe(egh.attach_coords).pr
ds = hp_to_latlon(ds, zoom)
if max(ds.longitude) > 180:
    ds = relon(ds).sortby("longitude")

H = mass_weighted_column_integral(calculate_mse(ds.ta,ds.zg,ds.hus))


Error: 

In [4]:
H.sel(time="2020-02-25").plot()

Error: 